# Assignment 4
## Description
In this assignment you must read in a file of metropolitan regions and associated sports teams from [assets/wikipedia_data.html](assets/wikipedia_data.html) and answer some questions about each metropolitan region. Each of these regions may have one or more teams from the "Big 4": NFL (football, in [assets/nfl.csv](assets/nfl.csv)), MLB (baseball, in [assets/mlb.csv](assets/mlb.csv)), NBA (basketball, in [assets/nba.csv](assets/nba.csv) or NHL (hockey, in [assets/nhl.csv](assets/nhl.csv)). Please keep in mind that all questions are from the perspective of the metropolitan region, and that this file is the "source of authority" for the location of a given sports team. Thus teams which are commonly known by a different area (e.g. "Oakland Raiders") need to be mapped into the metropolitan region given (e.g. San Francisco Bay Area). This will require some human data understanding outside of the data you've been given (e.g. you will have to hand-code some names, and might need to google to find out where teams are)!

For each sport I would like you to answer the question: **what is the win/loss ratio's correlation with the population of the city it is in?** Win/Loss ratio refers to the number of wins over the number of wins plus the number of losses. Remember that to calculate the correlation with [`pearsonr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), so you are going to send in two ordered lists of values, the populations from the wikipedia_data.html file and the win/loss ratio for a given sport in the same order. Average the win/loss ratios for those cities which have multiple teams of a single sport. Each sport is worth an equal amount in this assignment (20%\*4=80%) of the grade for this assignment. You should only use data **from year 2018** for your analysis -- this is important!

## Notes

1. Do not include data about the MLS or CFL in any of the work you are doing, we're only interested in the Big 4 in this assignment.
2. I highly suggest that you first tackle the four correlation questions in order, as they are all similar and worth the majority of grades for this assignment. This is by design!
3. It's fair game to talk with peers about high level strategy as well as the relationship between metropolitan areas and sports teams. However, do not post code solving aspects of the assignment (including such as dictionaries mapping areas to teams, or regexes which will clean up names).
4. There may be more teams than the assert statements test, remember to collapse multiple teams in one city into a single value!

As this assignment utilizes global variables in the skeleton code, to avoid having errors in your code you can either:

1. You can place all of your code within the function definitions for all of the questions (other than import statements).
2. You can create copies of all the global variables with the copy() method and proceed as usual.

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re


def nhl_correlation(): 
    # YOUR CODE HERE
    nhl_df=pd.read_csv("assets/nhl.csv")
    nhl_df = nhl_df[nhl_df['year']==2018]
    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]

    excl_list = ['Atlantic Division','Metropolitan Division','Central Division','Pacific Division']
    # exclude rows that look like headers
    nhl_df = nhl_df[~nhl_df['team'].isin(excl_list)]

    # remove * or special characters in the team name
    nhl_df['team'] = nhl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z\s]','',x))

    # rename columns
    cities.columns = ['city','population','NFL','MLB','NBA','NHL']

    # map team to city again
    team_city = {
        'Tampa Bay Lightning': 'Tampa Bay Area',
        'Boston Bruins': 'Boston',
        'Toronto Maple Leafs': 'Toronto',
        'Florida Panthers': 'Miami–Fort Lauderdale',
        'Detroit Red Wings': 'Detroit',
        'Montreal Canadiens': 'Montreal',
        'Ottawa Senators': 'Ottawa',
        'Buffalo Sabres': 'Buffalo',
        'Washington Capitals': 'Washington, D.C.',
        'Pittsburgh Penguins': 'Pittsburgh',
        'Philadelphia Flyers': 'Philadelphia',
        'Columbus Blue Jackets': 'Columbus',
        'New Jersey Devils': 'New York City',
        'Carolina Hurricanes': 'Raleigh',
        'New York Islanders': 'New York City',
        'New York Rangers': 'New York City',
        'Nashville Predators': 'Nashville',
        'Winnipeg Jets': 'Winnipeg',
        'Minnesota Wild': 'Minneapolis–Saint Paul',
        'Colorado Avalanche': 'Denver',
        'St Louis Blues': 'St. Louis',
        'Dallas Stars': 'Dallas–Fort Worth',
        'Chicago Blackhawks': 'Chicago',
        'Vegas Golden Knights': 'Las Vegas',
        'Anaheim Ducks': 'Los Angeles',
        'San Jose Sharks': 'San Francisco Bay Area',
        'Los Angeles Kings': 'Los Angeles',
        'Calgary Flames': 'Calgary',
        'Edmonton Oilers': 'Edmonton',
        'Vancouver Canucks': 'Vancouver',
        'Arizona Coyotes': 'Phoenix'
    }

    # convert the dict to a DataFrame
    team_city_df = pd.DataFrame(list(team_city.items()), columns = ['team','city'])

    nhl_df_city = nhl_df.merge(team_city_df, how = 'inner', left_on = 'team', right_on = 'team')

    nhl_df_city['W'] = nhl_df_city['W'].astype(int)
    nhl_df_city['L'] = nhl_df_city['L'].astype(int)
    nhl_df_city['w_l_ratio']=nhl_df_city['W']/(nhl_df_city['L'] + nhl_df_city['W'])

    nhl_df_city = nhl_df_city.groupby('city')['w_l_ratio'].mean().reset_index()


    nhl_df_city = nhl_df_city.merge(cities, how = 'left', left_on = 'city', right_on = 'city')
    # convert datatype
    nhl_df_city['population'] = nhl_df_city['population'].astype(int)
    population_by_region = nhl_df_city['population'] # pass in metropolitan area population from cities
    win_loss_by_region = nhl_df_city['w_l_ratio'] # pass in win/loss ratio from nhl_df in the same order as cities["Metropolitan area"]
    

    assert len(population_by_region) == len(win_loss_by_region), "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q1: There should be 28 teams being analysed for NHL"
    
    return stats.pearsonr(population_by_region, win_loss_by_region)[0]
    raise NotImplementedError()

In [2]:
nhl_correlation()

0.012486162921209909

In [3]:
#count_division = nhl_df['team'].value_counts()

In [4]:
#team_with_one_row = count_division[count_division ==1].index

In [5]:
#nhl_df[nhl_df['team'].isin(team_with_one_row)]

In [6]:
#excl_list = ['Atlantic Division','Metropolitan Division','Central Division','Pacific Division']

In [7]:
# exclude rows that look like headers
#nhl_df = nhl_df[~nhl_df['team'].isin(excl_list)]

In [8]:
# remove * or special characters in the team name
#nhl_df['team'] = nhl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z\s]','',x))

In [9]:
#nhl_df.head()

,team,GP,W,L,OL,PTS,PTS%,GF,GA,SRS,SOS,RPt%,ROW,year,League
1,Tampa Bay Lightning,82,54,23,5,113,.689,296,236,0.66,-0.07,.634,48,2018,NHL
2,Boston Bruins,82,50,20,12,112,.683,270,214,0.62,-0.07,.610,47,2018,NHL
3,Toronto Maple Leafs,82,49,26,7,105,.640,277,232,0.49,-0.06,.567,42,2018,NHL
4,Florida Panthers,82,44,30,8,96,.585,248,246,-0.01,-0.04,.537,41,2018,NHL
5,Detroit Red Wings,82,30,39,13,73,.445,217,255,-0.48,-0.01,.341,25,2018,NHL


In [10]:
#cities.columns = ['city','population','NFL','MLB','NBA','NHL']

In [11]:
#cities

,city,population,NFL,MLB,NBA,NHL
0,New York City,20153634,Giants Jets[note 1],Yankees Mets[note 2],Knicks Nets,Rangers Islanders Devils[note 3]
1,Los Angeles,13310447,Rams Chargers[note 4],Dodgers Angels,Lakers Clippers,Kings Ducks
2,San Francisco Bay Area,6657982,49ers Raiders[note 6],Giants Athletics,Warriors,Sharks[note 7]
3,Chicago,9512999,Bears[note 8],Cubs White Sox,Bulls[note 9],Blackhawks
4,Dallas–Fort Worth,7233323,Cowboys,Rangers,Mavericks,Stars
5,"Washington, D.C.",6131977,Redskins,Nationals[note 10],Wizards[note 11],Capitals
6,Philadelphia,6070500,Eagles,Phillies[note 12],76ers,Flyers[note 13]
7,Boston,4794447,Patriots[note 14],Red Sox[note 15],Celtics,Bruins
8,Minneapolis–Saint Paul,3551036,Vikings,Twins,Timberwolves,Wild[note 16]
9,Denver,2853077,Broncos,Rockies,Nuggets[note 17],Avalanche[note 18]


In [12]:
team_city = {
    'Tampa Bay Lightning': 'Tampa Bay Area',
    'Boston Bruins': 'Boston',
    'Toronto Maple Leafs': 'Toronto',
    'Florida Panthers': 'Miami–Fort Lauderdale',
    'Detroit Red Wings': 'Detroit',
    'Montreal Canadiens': 'Montreal',
    'Ottawa Senators': 'Ottawa',
    'Buffalo Sabres': 'Buffalo',
    'Washington Capitals': 'Washington, D.C.',
    'Pittsburgh Penguins': 'Pittsburgh',
    'Philadelphia Flyers': 'Philadelphia',
    'Columbus Blue Jackets': 'Columbus',
    'New Jersey Devils': 'New York City',
    'Carolina Hurricanes': 'Raleigh',
    'New York Islanders': 'New York City',
    'New York Rangers': 'New York City',
    'Nashville Predators': 'Nashville',
    'Winnipeg Jets': 'Winnipeg',
    'Minnesota Wild': 'Minneapolis–Saint Paul',
    'Colorado Avalanche': 'Denver',
    'St Louis Blues': 'St. Louis',
    'Dallas Stars': 'Dallas–Fort Worth',
    'Chicago Blackhawks': 'Chicago',
    'Vegas Golden Knights': 'Las Vegas',
    'Anaheim Ducks': 'Los Angeles',
    'San Jose Sharks': 'San Francisco Bay Area',
    'Los Angeles Kings': 'Los Angeles',
    'Calgary Flames': 'Calgary',
    'Edmonton Oilers': 'Edmonton',
    'Vancouver Canucks': 'Vancouver',
    'Arizona Coyotes': 'Phoenix'
}

In [13]:
# convert the dict to a DataFrame
#team_city_df = pd.DataFrame(list(team_city.items()), columns = ['team','city'])

In [14]:
#nhl_df_city = nhl_df.merge(team_city_df, how = 'inner', left_on = 'team', right_on = 'team')

In [15]:
#nhl_df_city['W'] = nhl_df_city['W'].astype(int)
#nhl_df_city['L'] = nhl_df_city['L'].astype(int)

In [16]:
#nhl_df_city = nhl_df_city.groupby('city').sum(['W','L'])

In [17]:
#nhl_df_city['w_l_ratio']=nhl_df_city['W']/nhl_df_city['L']

In [18]:
#nhl_df_city = nhl_df_city.reset_index()

In [19]:
#nhl_df_city = nhl_df_city.merge(cities, how = 'left', left_on = 'city', right_on = 'city')

In [20]:
#nhl_df_city.dtypes

city           object
W               int64
L               int64
year            int64
w_l_ratio     float64
population     object
NFL            object
MLB            object
NBA            object
NHL            object
dtype: object

## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def nba_correlation():
    # YOUR CODE HERE
    nba_df=pd.read_csv("assets/nba.csv")
    nba_df = nba_df[nba_df['year']==2018]
    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]

    #change column name and column datatype
    cities.columns = ['city', 'population', 'NFL', 'MLB', 'NBA', 'NHL']
    cities['population']= cities['population'].astype(int)

    # remove non characters 
    nba_df['team'] = nba_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z\s]','',x))
    # trim the string
    nba_df['team'] = nba_df['team'].str.strip()

    nba_to_city = {
        "Toronto Raptors": "Toronto",
        "Boston Celtics": "Boston",
        "Philadelphia ers": "Philadelphia",
        "Cleveland Cavaliers": "Cleveland",
        "Indiana Pacers": "Indianapolis",
        "Miami Heat": "Miami–Fort Lauderdale",
        "Milwaukee Bucks": "Milwaukee",
        "Washington Wizards": "Washington, D.C.",
        "Detroit Pistons": "Detroit",
        "Charlotte Hornets": "Charlotte",
        "New York Knicks": "New York City",
        "Brooklyn Nets": "New York City",
        "Chicago Bulls": "Chicago",
        "Orlando Magic": "Orlando",
        "Atlanta Hawks": "Atlanta",
        "Houston Rockets": "Houston",
        "Golden State Warriors": "San Francisco Bay Area",
        "Portland Trail Blazers": "Portland",
        "Oklahoma City Thunder": "Oklahoma City",
        "Utah Jazz": "Salt Lake City",
        "New Orleans Pelicans": "New Orleans",
        "San Antonio Spurs": "San Antonio",
        "Minnesota Timberwolves": "Minneapolis–Saint Paul",
        "Denver Nuggets": "Denver",
        "Los Angeles Clippers": "Los Angeles",
        "Los Angeles Lakers": "Los Angeles",
        "Sacramento Kings": "Sacramento",
        "Dallas Mavericks": "Dallas–Fort Worth",
        "Memphis Grizzlies": "Memphis",
        "Phoenix Suns": "Phoenix",
    }

    nba_city_df = pd.DataFrame(list(nba_to_city.items()), columns = ['team', 'city'])
    nba_df = nba_df.merge(nba_city_df, how = 'left', left_on = 'team', right_on = 'team')

    # convert datatype
    nba_df['W'] = nba_df['W'].astype(int)
    nba_df['L'] = nba_df['L'].astype(int)
    nba_df['w_l_ratio'] = nba_df['W']/(nba_df['W']+nba_df['L'])

    # group by city
    nba_df = nba_df.groupby('city')['w_l_ratio'].mean().reset_index()
    nba_df = nba_df.merge(cities, how = 'left', left_on = 'city', right_on = 'city')
    population_by_region = nba_df['population'] # pass in metropolitan area population from cities
    win_loss_by_region = nba_df['w_l_ratio'] # pass in win/loss ratio from nba_df in the same order as cities["Metropolitan area"]

    assert len(population_by_region) == len(win_loss_by_region), "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q2: There should be 28 teams being analysed for NBA"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]
    raise NotImplementedError()

In [4]:
nba_correlation()

-0.17657160252844614

In [23]:
#nba_df['team'] = nba_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z\s]','',x))

In [24]:
# trim the string
#nba_df['team'] = nba_df['team'].str.strip()

In [25]:
nba_to_city = {
    "Toronto Raptors": "Toronto",
    "Boston Celtics": "Boston",
    "Philadelphia ers": "Philadelphia",
    "Cleveland Cavaliers": "Cleveland",
    "Indiana Pacers": "Indianapolis",
    "Miami Heat": "Miami–Fort Lauderdale",
    "Milwaukee Bucks": "Milwaukee",
    "Washington Wizards": "Washington, D.C.",
    "Detroit Pistons": "Detroit",
    "Charlotte Hornets": "Charlotte",
    "New York Knicks": "New York City",
    "Brooklyn Nets": "New York City",
    "Chicago Bulls": "Chicago",
    "Orlando Magic": "Orlando",
    "Atlanta Hawks": "Atlanta",
    "Houston Rockets": "Houston",
    "Golden State Warriors": "San Francisco Bay Area",
    "Portland Trail Blazers": "Portland",
    "Oklahoma City Thunder": "Oklahoma City",
    "Utah Jazz": "Salt Lake City",
    "New Orleans Pelicans": "New Orleans",
    "San Antonio Spurs": "San Antonio",
    "Minnesota Timberwolves": "Minneapolis–Saint Paul",
    "Denver Nuggets": "Denver",
    "Los Angeles Clippers": "Los Angeles",
    "Los Angeles Lakers": "Los Angeles",
    "Sacramento Kings": "Sacramento",
    "Dallas Mavericks": "Dallas–Fort Worth",
    "Memphis Grizzlies": "Memphis",
    "Phoenix Suns": "Phoenix",
}

In [26]:
#nba_city_df = pd.DataFrame(list(nba_to_city.items()), columns = ['team', 'city'])

In [27]:
#nba_df = nba_df.merge(nba_city_df, how = 'left', left_on = 'team', right_on = 'team')

In [28]:
# convert datatype
#nba_df['W'] = nba_df['W'].astype(int)
#nba_df['L'] = nba_df['L'].astype(int)

In [29]:
#nba_df = nba_df.groupby('city').sum(['W','L'])

In [30]:
#cities.columns = ['city', 'population', 'NFL', 'MLB', 'NBA', 'NHL']

In [31]:
#cities['population']= cities['population'].astype(int)

In [32]:
#nba_df = nba_df.merge(cities, how = 'left', left_on = 'city', right_on = 'city')

In [33]:
#nba_df['w_l_ratio'] = nba_df['W']/(nba_df['W']+nba_df['L'])

In [34]:
#nba_df

,city,W,L,year,population,NFL,MLB,NBA,NHL,w_l_ratio
0,Atlanta,24,58,2018,5789700,Falcons,Braves,Hawks,[note 25],0.292683
1,Boston,55,27,2018,4794447,Patriots[note 14],Red Sox[note 15],Celtics,Bruins,0.670732
2,Charlotte,36,46,2018,2474314,Panthers,—,Hornets[note 49],—,0.439024
3,Chicago,27,55,2018,9512999,Bears[note 8],Cubs White Sox,Bulls[note 9],Blackhawks,0.329268
4,Cleveland,50,32,2018,2055612,Browns[note 29],Indians[note 30],Cavaliers[note 31],[note 32],0.609756
5,Dallas–Fort Worth,24,58,2018,7233323,Cowboys,Rangers,Mavericks,Stars,0.292683
6,Denver,46,36,2018,2853077,Broncos,Rockies,Nuggets[note 17],Avalanche[note 18],0.560976
7,Detroit,39,43,2018,4297617,Lions,Tigers[note 20],Pistons[note 21],Red Wings,0.475610
8,Houston,65,17,2018,6772470,Texans[note 24],Astros,Rockets,—,0.792683
9,Indianapolis,48,34,2018,2004230,Colts,[note 50],Pacers[note 51],—,0.585366


## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

In [5]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re


def mlb_correlation(): 
    # YOUR CODE HERE
    mlb_df=pd.read_csv("assets/mlb.csv")
    mlb_df = mlb_df[mlb_df['year']==2018]
    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]

    # change city column name and format 
    cities.columns = ['city', 'population', 'NFL', 'MLB', 'NBA', 'NHL']
    cities['population']= cities['population'].astype(int)

    mlb_team_city= {
        "Boston Red Sox": "Boston",
        "New York Yankees": "New York City",
        "Tampa Bay Rays": "Tampa Bay Area",
        "Toronto Blue Jays": "Toronto",
        "Baltimore Orioles": "Baltimore",
        "Cleveland Indians": "Cleveland",
        "Minnesota Twins": "Minneapolis–Saint Paul",
        "Detroit Tigers": "Detroit",
        "Chicago White Sox": "Chicago",
        "Kansas City Royals": "Kansas City",
        "Houston Astros": "Houston",
        "Oakland Athletics": "San Francisco Bay Area",
        "Seattle Mariners": "Seattle",
        "Los Angeles Angels": "Los Angeles",
        "Texas Rangers": "Dallas–Fort Worth",
        "Atlanta Braves": "Atlanta",
        "Washington Nationals": "Washington, D.C.",
        "Philadelphia Phillies": "Philadelphia",
        "New York Mets": "New York City",
        "Miami Marlins": "Miami–Fort Lauderdale",
        "Milwaukee Brewers": "Milwaukee",
        "Chicago Cubs": "Chicago",
        "St. Louis Cardinals": "St. Louis",
        "Pittsburgh Pirates": "Pittsburgh",
        "Cincinnati Reds": "Cincinnati",
        "Los Angeles Dodgers": "Los Angeles",
        "Colorado Rockies": "Denver",
        "Arizona Diamondbacks": "Phoenix",
        "San Francisco Giants": "San Francisco Bay Area",
        "San Diego Padres": "San Diego",
    }
    mbl_team_city_df = pd.DataFrame(list(mlb_team_city.items()), columns=['team','city'])

    mlb_df = mlb_df.merge(mbl_team_city_df, how = 'left', on = 'team')
    mlb_df['w_l_ratio'] = mlb_df['W']/(mlb_df['W'] + mlb_df['L'])
    mlb_df = mlb_df.groupby('city').mean()['w_l_ratio'].reset_index()

    mlb_df = mlb_df.merge(cities, how = 'left', on = 'city')
     
    population_by_region = mlb_df['population'] # pass in metropolitan area population from cities
    win_loss_by_region = mlb_df['w_l_ratio'] # pass in win/loss ratio from mlb_df in the same order as cities["Metropolitan area"]

    assert len(population_by_region) == len(win_loss_by_region), "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, "Q3: There should be 26 teams being analysed for MLB"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]
    raise NotImplementedError()

In [6]:
mlb_correlation()

0.15027698302669307

In [37]:
mlb_team_city= {
    "Boston Red Sox": "Boston",
    "New York Yankees": "New York City",
    "Tampa Bay Rays": "Tampa Bay Area",
    "Toronto Blue Jays": "Toronto",
    "Baltimore Orioles": "Baltimore",
    "Cleveland Indians": "Cleveland",
    "Minnesota Twins": "Minneapolis–Saint Paul",
    "Detroit Tigers": "Detroit",
    "Chicago White Sox": "Chicago",
    "Kansas City Royals": "Kansas City",
    "Houston Astros": "Houston",
    "Oakland Athletics": "San Francisco Bay Area",
    "Seattle Mariners": "Seattle",
    "Los Angeles Angels": "Los Angeles",
    "Texas Rangers": "Dallas–Fort Worth",
    "Atlanta Braves": "Atlanta",
    "Washington Nationals": "Washington, D.C.",
    "Philadelphia Phillies": "Philadelphia",
    "New York Mets": "New York City",
    "Miami Marlins": "Miami–Fort Lauderdale",
    "Milwaukee Brewers": "Milwaukee",
    "Chicago Cubs": "Chicago",
    "St. Louis Cardinals": "St. Louis",
    "Pittsburgh Pirates": "Pittsburgh",
    "Cincinnati Reds": "Cincinnati",
    "Los Angeles Dodgers": "Los Angeles",
    "Colorado Rockies": "Denver",
    "Arizona Diamondbacks": "Phoenix",
    "San Francisco Giants": "San Francisco Bay Area",
    "San Diego Padres": "San Diego",
}

In [38]:
#mbl_team_city_df = pd.DataFrame(list(mlb_team_city.items()), columns=['team','city'])

In [39]:
#mlb_df = mlb_df.merge(mbl_team_city_df, how = 'left', on = 'team')

In [40]:
#mlb_df = mlb_df.groupby('city').sum(['W','L'])

In [41]:
#cities.columns = ['city', 'population', 'NFL', 'MLB', 'NBA', 'NHL']
#cities['population']= cities['population'].astype(int)

In [42]:
#mlb_df = mlb_df.merge(cities, how = 'left', on = 'city')

In [43]:
#mlb_df

,city,W,L,W-L%,year,population,NFL,MLB,NBA,NHL,w_l_ratio
0,Atlanta,90,72,0.556,2018,5789700,Falcons,Braves,Hawks,[note 25],0.555556
1,Baltimore,47,115,0.290,2018,2798886,Ravens[note 45],Orioles[note 46],[note 47],—,0.290123
2,Boston,108,54,0.667,2018,4794447,Patriots[note 14],Red Sox[note 15],Celtics,Bruins,0.666667
3,Chicago,157,168,0.966,4036,9512999,Bears[note 8],Cubs White Sox,Bulls[note 9],Blackhawks,0.483077
4,Cincinnati,67,95,0.414,2018,2165139,Bengals,Reds[note 35],[note 36],—,0.413580
5,Cleveland,91,71,0.562,2018,2055612,Browns[note 29],Indians[note 30],Cavaliers[note 31],[note 32],0.561728
6,Dallas–Fort Worth,67,95,0.414,2018,7233323,Cowboys,Rangers,Mavericks,Stars,0.413580
7,Denver,91,72,0.558,2018,2853077,Broncos,Rockies,Nuggets[note 17],Avalanche[note 18],0.558282
8,Detroit,64,98,0.395,2018,4297617,Lions,Tigers[note 20],Pistons[note 21],Red Wings,0.395062
9,Houston,103,59,0.636,2018,6772470,Texans[note 24],Astros,Rockets,—,0.635802


## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

In [7]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def nfl_correlation(): 
    # YOUR CODE HERE   
    nfl_df=pd.read_csv("assets/nfl.csv")
# filter only 2018
    nfl_df = nfl_df[nfl_df['year']==2018]
    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]
    #change column name and column datatype
    cities.columns = ['city', 'population', 'NFL', 'MLB', 'NBA', 'NHL']
    cities['population']= cities['population'].astype(int)
    nlf_excl_list = ['AFC East','AFC North','AFC South','AFC West','NFC East','NFC North','NFC South','NFC West']
    # remove weird looking rows
    nfl_df = nfl_df[~nfl_df['team'].isin(nlf_excl_list)]

    # cleanup team name
    nfl_df['team'] = nfl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))
    #remove space before and after the string
    nfl_df['team'] = nfl_df['team'].str.strip()

    nfl_team_to_city = {
        "New England Patriots": "Boston",
        "Miami Dolphins": "Miami–Fort Lauderdale",
        "Buffalo Bills": "Buffalo",
        "New York Jets": "New York City",
        "Baltimore Ravens": "Baltimore",
        "Pittsburgh Steelers": "Pittsburgh",
        "Cleveland Browns": "Cleveland",
        "Cincinnati Bengals": "Cincinnati",
        "Houston Texans": "Houston",
        "Indianapolis Colts": "Indianapolis",
        "Tennessee Titans": "Nashville",
        "Jacksonville Jaguars": "Jacksonville",
        "Kansas City Chiefs": "Kansas City",
        "Los Angeles Chargers": "Los Angeles",
        "Denver Broncos": "Denver",
        "Oakland Raiders": "San Francisco Bay Area",
        "Dallas Cowboys": "Dallas–Fort Worth",
        "Philadelphia Eagles": "Philadelphia",
        "Washington Redskins": "Washington, D.C.",
        "New York Giants": "New York City",
        "Chicago Bears": "Chicago",
        "Minnesota Vikings": "Minneapolis–Saint Paul",
        "Green Bay Packers": "Green Bay",
        "Detroit Lions": "Detroit",
        "New Orleans Saints": "New Orleans",
        "Carolina Panthers": "Charlotte",
        "Atlanta Falcons": "Atlanta",
        "Tampa Bay Buccaneers": "Tampa Bay Area",
        "Los Angeles Rams": "Los Angeles",
        "Seattle Seahawks": "Seattle",
        "San Francisco 49ers": "San Francisco Bay Area",
        "Arizona Cardinals": "Phoenix"
    }

    nfl_team_city = pd.DataFrame(list(nfl_team_to_city.items()), columns = ['team', 'city'])
    nfl_df = nfl_df.merge(nfl_team_city, how = 'left', on = 'team')

    nfl_df['W']= nfl_df['W'].astype(int)
    nfl_df['L'] = nfl_df['L'].astype(int)
    nfl_df['w_l_ratio']= nfl_df['W'] / (nfl_df['W']+nfl_df['L'])

    nfl_df = nfl_df.groupby('city').mean()['w_l_ratio'].reset_index()
    nfl_df = nfl_df.merge(cities, how = 'left', on = 'city')

    population_by_region = nfl_df['population'] # pass in metropolitan area population from cities
    win_loss_by_region = nfl_df['w_l_ratio'] # pass in win/loss ratio from nfl_df in the same order as cities["Metropolitan area"]

    assert len(population_by_region) == len(win_loss_by_region), "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, "Q4: There should be 29 teams being analysed for NFL"

    return stats.pearsonr(population_by_region, win_loss_by_region)[0]
    raise NotImplementedError()

In [8]:
nfl_correlation()

0.004922112149349409

In [46]:
#nfl_df = nfl_df[nfl_df['year']==2018]

In [47]:
#nlf_excl_list = ['AFC East','AFC North','AFC South','AFC West','NFC East','NFC North','NFC South','NFC West']

In [48]:
# remove weird looking rows
#nfl_df = nfl_df[~nfl_df['team'].isin(nlf_excl_list)]

KeyError: 'team'

In [ ]:
# cleanup team name
#nfl_df['team'] = nfl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

In [ ]:
#remove space before and after the string
#nfl_df['team'] = nfl_df['team'].str.strip()

In [ ]:
nfl_team_to_city = {
    "New England Patriots": "Boston",
    "Miami Dolphins": "Miami–Fort Lauderdale",
    "Buffalo Bills": "Buffalo",
    "New York Jets": "New York City",
    "Baltimore Ravens": "Baltimore",
    "Pittsburgh Steelers": "Pittsburgh",
    "Cleveland Browns": "Cleveland",
    "Cincinnati Bengals": "Cincinnati",
    "Houston Texans": "Houston",
    "Indianapolis Colts": "Indianapolis",
    "Tennessee Titans": "Nashville",
    "Jacksonville Jaguars": "Jacksonville",
    "Kansas City Chiefs": "Kansas City",
    "Los Angeles Chargers": "Los Angeles",
    "Denver Broncos": "Denver",
    "Oakland Raiders": "San Francisco Bay Area",
    "Dallas Cowboys": "Dallas–Fort Worth",
    "Philadelphia Eagles": "Philadelphia",
    "Washington Redskins": "Washington, D.C.",
    "New York Giants": "New York City",
    "Chicago Bears": "Chicago",
    "Minnesota Vikings": "Minneapolis–Saint Paul",
    "Green Bay Packers": "Green Bay",
    "Detroit Lions": "Detroit",
    "New Orleans Saints": "New Orleans",
    "Carolina Panthers": "Charlotte",
    "Atlanta Falcons": "Atlanta",
    "Tampa Bay Buccaneers": "Tampa Bay Area",
    "Los Angeles Rams": "Los Angeles",
    "Seattle Seahawks": "Seattle",
    "San Francisco 49ers": "San Francisco Bay Area",
    "Arizona Cardinals": "Phoenix"
}

In [ ]:
#nfl_team_city = pd.DataFrame(list(nfl_team_to_city.items()), columns = ['team', 'city'])

In [ ]:
#nfl_team_city

In [ ]:
#nfl_df = nfl_df.merge(nfl_team_city, how = 'left', on = 'team')

In [ ]:
#nfl_df.head()

In [ ]:
#nfl_df['W']= nfl_df['W'].astype(int)
#nfl_df['L'] = nfl_df['L'].astype(int)

In [ ]:
#nfl_df.dtypes

In [ ]:
#change column name and column datatype
#cities.columns = ['city', 'population', 'NFL', 'MLB', 'NBA', 'NHL']
#cities['population']= cities['population'].astype(int)

In [ ]:
#nfl_df = nfl_df.groupby('city').sum(['W','L'])

In [ ]:
#nfl_df = nfl_df.merge(cities, how = 'left', on = 'city')

In [ ]:
#nfl_df.head()

In [ ]:
#nfl_df['w_l_ratio']= nfl_df['W'] / (nfl_df['W']+nfl_df['L'])

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate. This question is worth 20% of the grade for this assignment.

In [190]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

def sports_team_performance():
    # YOUR CODE HERE
    mlb_df=pd.read_csv("assets/mlb.csv")
    nhl_df=pd.read_csv("assets/nhl.csv")
    nba_df=pd.read_csv("assets/nba.csv")
    nfl_df=pd.read_csv("assets/nfl.csv")
    mlb_df = mlb_df[mlb_df['year'] == 2018]
    nhl_df = nhl_df[nhl_df['year'] == 2018]
    nba_df = nba_df[nba_df['year'] == 2018]
    nfl_df = nfl_df[nfl_df['year'] == 2018]
    cities=pd.read_html("assets/wikipedia_data.html")[1]
    cities=cities.iloc[:-1,[0,3,5,6,7,8]]
    cities.columns = ['city', 'population','NFL','MLB','NBA','NHL']
    cities['population']=cities['population'].astype(int)
    cities = cities[['city','population']]
    
    #clean team name
    nfl_df['team'] = nfl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]','',x))
    nlf_excl_list = ['AFC East','AFC North','AFC South','AFC West','NFC East','NFC North','NFC South','NFC West']
    #excl weird rows
    nfl_df = nfl_df[~nfl_df['team'].isin(nlf_excl_list)]
    
    nfl_df['W'] = nfl_df['W'].astype(int)
    nfl_df['L']= nfl_df['L'].astype(int)
    nfl_df['NFL'] = nfl_df['W-L%'].astype(float) 
    
    nfl_team_to_city = {
        "New England Patriots": "Boston",
        "Miami Dolphins": "Miami–Fort Lauderdale",
        "Buffalo Bills": "Buffalo",
        "New York Jets": "New York City",
        "Baltimore Ravens": "Baltimore",
        "Pittsburgh Steelers": "Pittsburgh",
        "Cleveland Browns": "Cleveland",
        "Cincinnati Bengals": "Cincinnati",
        "Houston Texans": "Houston",
        "Indianapolis Colts": "Indianapolis",
        "Tennessee Titans": "Nashville",
        "Jacksonville Jaguars": "Jacksonville",
        "Kansas City Chiefs": "Kansas City",
        "Los Angeles Chargers": "Los Angeles",
        "San Diego Chargers": "San Diego",
        "Denver Broncos": "Denver",
        "Oakland Raiders": "San Francisco Bay Area",
        "Dallas Cowboys": "Dallas–Fort Worth",
        "Philadelphia Eagles": "Philadelphia",
        "Washington Redskins": "Washington, D.C.",
        "New York Giants": "New York City",
        "Chicago Bears": "Chicago",
        "Minnesota Vikings": "Minneapolis–Saint Paul",
        "Green Bay Packers": "Green Bay",
        "Detroit Lions": "Detroit",
        "New Orleans Saints": "New Orleans",
        "Carolina Panthers": "Charlotte",
        "Atlanta Falcons": "Atlanta",
        "Tampa Bay Buccaneers": "Tampa Bay Area",
        "Los Angeles Rams": "Los Angeles",
        "St Louis Rams": "St. Louis",
        "Seattle Seahawks": "Seattle",
        "San Francisco 49ers": "San Francisco Bay Area",
        "Arizona Cardinals": "Phoenix",
    }    
    
    nfl_team_to_city_df = pd.DataFrame(list(nfl_team_to_city.items()), columns = ['team','city'])
    base_df = nfl_df.merge(nfl_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')
    
    base_df = base_df.groupby(['city', 'year']).mean()['NFL'].reset_index()
    
    
    mlb_team_to_city = {
        "Boston Red Sox": "Boston",
        "New York Yankees": "New York City",
        "Tampa Bay Rays": "Tampa Bay Area",
        "Toronto Blue Jays": "Toronto",
        "Baltimore Orioles": "Baltimore",
        "Cleveland Indians": "Cleveland",
        "Minnesota Twins": "Minneapolis–Saint Paul",
        "Detroit Tigers": "Detroit",
        "Chicago White Sox": "Chicago",
        "Kansas City Royals": "Kansas City",
        "Houston Astros": "Houston",
        "Oakland Athletics": "San Francisco Bay Area",
        "Seattle Mariners": "Seattle",
        "Los Angeles Angels": "Los Angeles",
        "Los Angeles Angels of Anaheim": "Los Angeles",
        "Texas Rangers": "Dallas–Fort Worth",
        "Atlanta Braves": "Atlanta",
        "Washington Nationals": "Washington, D.C.",
        "Philadelphia Phillies": "Philadelphia",
        "New York Mets": "New York City",
        "Miami Marlins": "Miami–Fort Lauderdale",
        "Milwaukee Brewers": "Milwaukee",
        "Chicago Cubs": "Chicago",
        "St. Louis Cardinals": "St. Louis",
        "Pittsburgh Pirates": "Pittsburgh",
        "Cincinnati Reds": "Cincinnati",
        "Los Angeles Dodgers": "Los Angeles",
        "Colorado Rockies": "Denver",
        "Arizona Diamondbacks": "Phoenix",
        "San Francisco Giants": "San Francisco Bay Area",
        "San Diego Padres": "San Diego",
    }
    
    mlb_team_to_city_df = pd.DataFrame(list(mlb_team_to_city.items()), columns = ['team', 'city'])
    mlb_df = mlb_df.merge(mlb_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')
    mlb_df['MLB']= mlb_df['W-L%']
    mlb_df = mlb_df.groupby(['city','year']).mean()['MLB'].reset_index()
    
    #clean name
    nhl_df['team']=nhl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]','',x))
    excl_list = ['Atlantic Division','Metropolitan Division','Central Division','Pacific Division']
    nhl_df = nhl_df[~nhl_df['team'].isin(excl_list)]
    nhl_df['W'] = nhl_df['W'].astype(int)
    nhl_df['L'] = nhl_df['L'].astype(int)
    nhl_df['NHL']=nhl_df['W']/(nhl_df['W'] + nhl_df['L'])
    
    nhl_team_to_city = {
        "Tampa Bay Lightning": "Tampa Bay Area",
        "Boston Bruins": "Boston",
        "Toronto Maple Leafs": "Toronto",
        "Florida Panthers": "Miami–Fort Lauderdale",
        "Detroit Red Wings": "Detroit",
        "Montreal Canadiens": "Montreal",
        "Ottawa Senators": "Ottawa",
        "Buffalo Sabres": "Buffalo",
        "Washington Capitals": "Washington, D.C.",
        "Pittsburgh Penguins": "Pittsburgh",
        "Philadelphia Flyers": "Philadelphia",
        "Columbus Blue Jackets": "Columbus",
        "New Jersey Devils": "New York City",
        "Carolina Hurricanes": "Raleigh",
        "New York Islanders": "New York City",
        "New York Rangers": "New York City",
        "Nashville Predators": "Nashville",
        "Winnipeg Jets": "Winnipeg",
        "Minnesota Wild": "Minneapolis–Saint Paul",
        "Colorado Avalanche": "Denver",
        "St Louis Blues": "St. Louis",
        "Dallas Stars": "Dallas–Fort Worth",
        "Chicago Blackhawks": "Chicago",
        "Vegas Golden Knights": "Las Vegas",
        "Anaheim Ducks": "Los Angeles",
        "San Jose Sharks": "San Francisco Bay Area",
        "Los Angeles Kings": "Los Angeles",
        "Calgary Flames": "Calgary",
        "Edmonton Oilers": "Edmonton",
        "Vancouver Canucks": "Vancouver",
        "Arizona Coyotes": "Phoenix",
        "Phoenix Coyotes": "Phoenix",
    }
    
    nhl_team_to_city_df = pd.DataFrame(list(nhl_team_to_city.items()), columns = ['team','city'])
    
    nhl_df = nhl_df.merge(nhl_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')
    nhl_df = nhl_df.groupby(['city','year']).mean()['NHL'].reset_index()
    
    #nba
    nba_df['team'] = nba_df['team'].str.split('*').str[0].str.split('(').str[0].str.strip()
    
    nba_team_to_city = {
        "Toronto Raptors": "Toronto",
        "Boston Celtics": "Boston",
        "Philadelphia 76ers": "Philadelphia",
        "Cleveland Cavaliers": "Cleveland",
        "Indiana Pacers": "Indianapolis",
        "Miami Heat": "Miami–Fort Lauderdale",
        "Milwaukee Bucks": "Milwaukee",
        "Washington Wizards": "Washington, D.C.",
        "Detroit Pistons": "Detroit",
        "Charlotte Hornets": "Charlotte",
        "Charlotte Bobcats": "Charlotte",
        "New York Knicks": "New York City",
        "Brooklyn Nets": "New York City",
        "Chicago Bulls": "Chicago",
        "Orlando Magic": "Orlando",
        "Atlanta Hawks": "Atlanta",
        "Houston Rockets": "Houston",
        "Golden State Warriors": "San Francisco Bay Area",
        "Portland Trail Blazers": "Portland",
        "Oklahoma City Thunder": "Oklahoma City",
        "Utah Jazz": "Salt Lake City",
        "New Orleans Pelicans": "New Orleans",
        "San Antonio Spurs": "San Antonio",
        "Minnesota Timberwolves": "Minneapolis–Saint Paul",
        "Denver Nuggets": "Denver",
        "Los Angeles Clippers": "Los Angeles",
        "Los Angeles Lakers": "Los Angeles",
        "Sacramento Kings": "Sacramento",
        "Dallas Mavericks": "Dallas–Fort Worth",
        "Memphis Grizzlies": "Memphis",
        "Phoenix Suns": "Phoenix",
    }
    nba_team_to_city_df = pd.DataFrame(list(nba_team_to_city.items()), columns = ['team','city'])
    nba_df = nba_df.merge(nba_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')
    
    nba_excl_list = ['Atlantic Division', 'Central Division','Southeast Division','Northwest Division','Pacific Division',
                    'Southwest Division']
    nba_df = nba_df[~nba_df['team'].isin(nba_excl_list)]
    nba_df['W/L%']=nba_df['W/L%'].astype(float)
    nba_df['NBA'] = nba_df['W/L%']
    nba_df=nba_df.groupby(['city','year']).mean()['NBA'].reset_index()
    
    # merge with cities
    # base_df = base_df.merge(cities, how = 'outer', on = 'city')
    
    # merge with MLB
    # base_df = base_df.merge(mlb_df, how = 'outer', on = ['city', 'year'])
    # merge with nhl_df
    # base_df = base_df.merge(nhl_df, how = 'outer', on = ['city', 'year'])
    
    # merge with nba_df
    # base_df = base_df.merge(nba_df, how = 'outer', on = ['city', 'year'])
    
    
    # Note: p_values is a full dataframe, so df.loc["NFL","NBA"] should be the same as df.loc["NBA","NFL"] and
    # df.loc["NFL","NFL"] should return np.nan
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    sports_dict = {'NFL': base_df, 'NBA': nba_df, 'NHL': nhl_df, 'MLB': mlb_df}
    p_values = pd.DataFrame({k:np.nan for k in sports}, index=sports)    
    for sport1 in sports:
        for sport2 in sports:
            if sport2!= sport1:
                df1 = sports_dict[sport1]
                df2 = sports_dict[sport2]
                df = pd.merge(df1, df2, on='city', how='inner')
                
                p_val = stats.ttest_rel(df[sport1], df[sport2])[1]
                p_values.loc[sport1, sport2] = p_val

    
    assert abs(p_values.loc["NBA", "NHL"] - 0.02) <= 1e-2, "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc["MLB", "NFL"] - 0.80) <= 1e-2, "The MLB-NFL p-value should be around 0.80"
    return p_values
    raise NotImplementedError()

In [138]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df=pd.read_csv("assets/mlb.csv")
nhl_df=pd.read_csv("assets/nhl.csv")
nba_df=pd.read_csv("assets/nba.csv")
nfl_df=pd.read_csv("assets/nfl.csv")
cities=pd.read_html("assets/wikipedia_data.html")[1]
cities=cities.iloc[:-1,[0,3,5,6,7,8]]

In [139]:
nfl_df['team'] = nfl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]','',x))

In [140]:
nlf_excl_list = ['AFC East','AFC North','AFC South','AFC West','NFC East','NFC North','NFC South','NFC West']
nfl_df = nfl_df[~nfl_df['team'].isin(nlf_excl_list)]

In [141]:
print(nfl_df['team'].tolist())

['New England Patriots', 'Miami Dolphins', 'Buffalo Bills', 'New York Jets', 'Baltimore Ravens', 'Pittsburgh Steelers', 'Cleveland Browns', 'Cincinnati Bengals', 'Houston Texans', 'Indianapolis Colts', 'Tennessee Titans', 'Jacksonville Jaguars', 'Kansas City Chiefs', 'Los Angeles Chargers', 'Denver Broncos', 'Oakland Raiders', 'Dallas Cowboys', 'Philadelphia Eagles', 'Washington Redskins', 'New York Giants', 'Chicago Bears', 'Minnesota Vikings', 'Green Bay Packers', 'Detroit Lions', 'New Orleans Saints', 'Carolina Panthers', 'Atlanta Falcons', 'Tampa Bay Buccaneers', 'Los Angeles Rams', 'Seattle Seahawks', 'San Francisco 49ers', 'Arizona Cardinals', 'New England Patriots', 'Buffalo Bills', 'Miami Dolphins', 'New York Jets', 'Pittsburgh Steelers', 'Baltimore Ravens', 'Cincinnati Bengals', 'Cleveland Browns', 'Jacksonville Jaguars', 'Tennessee Titans', 'Houston Texans', 'Indianapolis Colts', 'Kansas City Chiefs', 'Los Angeles Chargers', 'Oakland Raiders', 'Denver Broncos', 'Philadelphia 

In [142]:
nfl_df['W'] = nfl_df['W'].astype(int)
nfl_df['L']= nfl_df['L'].astype(int)
nfl_df['NFL'] = nfl_df['W-L%'].astype(float)

In [143]:
nfl_team_to_city = {
    "New England Patriots": "Boston",
    "Miami Dolphins": "Miami–Fort Lauderdale",
    "Buffalo Bills": "Buffalo",
    "New York Jets": "New York City",
    "Baltimore Ravens": "Baltimore",
    "Pittsburgh Steelers": "Pittsburgh",
    "Cleveland Browns": "Cleveland",
    "Cincinnati Bengals": "Cincinnati",
    "Houston Texans": "Houston",
    "Indianapolis Colts": "Indianapolis",
    "Tennessee Titans": "Nashville",
    "Jacksonville Jaguars": "Jacksonville",
    "Kansas City Chiefs": "Kansas City",
    "Los Angeles Chargers": "Los Angeles",
    "San Diego Chargers": "San Diego",
    "Denver Broncos": "Denver",
    "Oakland Raiders": "San Francisco Bay Area",
    "Dallas Cowboys": "Dallas–Fort Worth",
    "Philadelphia Eagles": "Philadelphia",
    "Washington Redskins": "Washington, D.C.",
    "New York Giants": "New York City",
    "Chicago Bears": "Chicago",
    "Minnesota Vikings": "Minneapolis–Saint Paul",
    "Green Bay Packers": "Green Bay",
    "Detroit Lions": "Detroit",
    "New Orleans Saints": "New Orleans",
    "Carolina Panthers": "Charlotte",
    "Atlanta Falcons": "Atlanta",
    "Tampa Bay Buccaneers": "Tampa Bay Area",
    "Los Angeles Rams": "Los Angeles",
    "St Louis Rams": "St. Louis",
    "Seattle Seahawks": "Seattle",
    "San Francisco 49ers": "San Francisco Bay Area",
    "Arizona Cardinals": "Phoenix",
}

In [144]:
cities.columns = ['city', 'population','NFL','MLB','NBA','NHL']
cities['population']=cities['population'].astype(int)

In [145]:
cities = cities[['city','population']]

In [146]:
nfl_team_to_city_df = pd.DataFrame(list(nfl_team_to_city.items()), columns = ['team','city'])

In [147]:
base_df = nfl_df.merge(nfl_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')

In [148]:
base_df = base_df.groupby(['city', 'year']).mean()['NFL'].reset_index()

In [132]:
base_df

,city,year,NFL
0,Atlanta,2014,0.375
1,Atlanta,2015,0.500
2,Atlanta,2016,0.688
3,Atlanta,2017,0.625
4,Atlanta,2018,0.438
...,...,...,...
143,"Washington, D.C.",2014,0.250
144,"Washington, D.C.",2015,0.563
145,"Washington, D.C.",2016,0.531
146,"Washington, D.C.",2017,0.438


In [149]:
mlb_df

,team,W,L,W-L%,GB,year,League
0,Boston Red Sox,108,54,0.667,--,2018,MLB
1,New York Yankees,100,62,0.617,8.0,2018,MLB
2,Tampa Bay Rays,90,72,0.556,18.0,2018,MLB
3,Toronto Blue Jays,73,89,0.451,35.0,2018,MLB
4,Baltimore Orioles,47,115,0.290,61.0,2018,MLB
...,...,...,...,...,...,...,...
145,Los Angeles Dodgers,94,68,0.580,--,2014,MLB
146,San Francisco Giants,88,74,0.543,6.0,2014,MLB
147,San Diego Padres,77,85,0.475,17.0,2014,MLB
148,Colorado Rockies,66,96,0.407,28.0,2014,MLB


In [150]:
mlb_team_to_city = {
    "Boston Red Sox": "Boston",
    "New York Yankees": "New York City",
    "Tampa Bay Rays": "Tampa Bay Area",
    "Toronto Blue Jays": "Toronto",
    "Baltimore Orioles": "Baltimore",
    "Cleveland Indians": "Cleveland",
    "Minnesota Twins": "Minneapolis–Saint Paul",
    "Detroit Tigers": "Detroit",
    "Chicago White Sox": "Chicago",
    "Kansas City Royals": "Kansas City",
    "Houston Astros": "Houston",
    "Oakland Athletics": "San Francisco Bay Area",
    "Seattle Mariners": "Seattle",
    "Los Angeles Angels": "Los Angeles",
    "Los Angeles Angels of Anaheim": "Los Angeles",
    "Texas Rangers": "Dallas–Fort Worth",
    "Atlanta Braves": "Atlanta",
    "Washington Nationals": "Washington, D.C.",
    "Philadelphia Phillies": "Philadelphia",
    "New York Mets": "New York City",
    "Miami Marlins": "Miami–Fort Lauderdale",
    "Milwaukee Brewers": "Milwaukee",
    "Chicago Cubs": "Chicago",
    "St. Louis Cardinals": "St. Louis",
    "Pittsburgh Pirates": "Pittsburgh",
    "Cincinnati Reds": "Cincinnati",
    "Los Angeles Dodgers": "Los Angeles",
    "Colorado Rockies": "Denver",
    "Arizona Diamondbacks": "Phoenix",
    "San Francisco Giants": "San Francisco Bay Area",
    "San Diego Padres": "San Diego",
}

In [151]:
mlb_team_to_city_df = pd.DataFrame(list(mlb_team_to_city.items()), columns = ['team', 'city'])

In [152]:
mlb_df = mlb_df.merge(mlb_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')

In [153]:
mlb_df['MLB']= mlb_df['W-L%']

In [154]:
mlb_df = mlb_df.groupby(['city','year']).mean()['MLB'].reset_index()

In [155]:
mlb_df

,city,year,MLB
0,Atlanta,2014,0.488
1,Atlanta,2015,0.414
2,Atlanta,2016,0.422
3,Atlanta,2017,0.444
4,Atlanta,2018,0.556
...,...,...,...
125,"Washington, D.C.",2014,0.593
126,"Washington, D.C.",2015,0.512
127,"Washington, D.C.",2016,0.586
128,"Washington, D.C.",2017,0.599


In [156]:
nhl_df['team']=nhl_df['team'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]','',x))

In [157]:
excl_list = ['Atlantic Division','Metropolitan Division','Central Division','Pacific Division']
nhl_df = nhl_df[~nhl_df['team'].isin(excl_list)]

In [158]:
print(nhl_df['team'].tolist())

['Tampa Bay Lightning', 'Boston Bruins', 'Toronto Maple Leafs', 'Florida Panthers', 'Detroit Red Wings', 'Montreal Canadiens', 'Ottawa Senators', 'Buffalo Sabres', 'Washington Capitals', 'Pittsburgh Penguins', 'Philadelphia Flyers', 'Columbus Blue Jackets', 'New Jersey Devils', 'Carolina Hurricanes', 'New York Islanders', 'New York Rangers', 'Nashville Predators', 'Winnipeg Jets', 'Minnesota Wild', 'Colorado Avalanche', 'St Louis Blues', 'Dallas Stars', 'Chicago Blackhawks', 'Vegas Golden Knights', 'Anaheim Ducks', 'San Jose Sharks', 'Los Angeles Kings', 'Calgary Flames', 'Edmonton Oilers', 'Vancouver Canucks', 'Arizona Coyotes', 'Montreal Canadiens', 'Ottawa Senators', 'Boston Bruins', 'Toronto Maple Leafs', 'Tampa Bay Lightning', 'Florida Panthers', 'Detroit Red Wings', 'Buffalo Sabres', 'Washington Capitals', 'Pittsburgh Penguins', 'Columbus Blue Jackets', 'New York Rangers', 'New York Islanders', 'Philadelphia Flyers', 'Carolina Hurricanes', 'New Jersey Devils', 'Chicago Blackhawks

In [159]:
nhl_df['W'] = nhl_df['W'].astype(int)
nhl_df['L'] = nhl_df['L'].astype(int)
nhl_df['NHL']=nhl_df['W']/(nhl_df['W'] + nhl_df['L'])

In [160]:
nhl_team_to_city = {
    "Tampa Bay Lightning": "Tampa Bay Area",
    "Boston Bruins": "Boston",
    "Toronto Maple Leafs": "Toronto",
    "Florida Panthers": "Miami–Fort Lauderdale",
    "Detroit Red Wings": "Detroit",
    "Montreal Canadiens": "Montreal",
    "Ottawa Senators": "Ottawa",
    "Buffalo Sabres": "Buffalo",
    "Washington Capitals": "Washington, D.C.",
    "Pittsburgh Penguins": "Pittsburgh",
    "Philadelphia Flyers": "Philadelphia",
    "Columbus Blue Jackets": "Columbus",
    "New Jersey Devils": "New York City",
    "Carolina Hurricanes": "Raleigh",
    "New York Islanders": "New York City",
    "New York Rangers": "New York City",
    "Nashville Predators": "Nashville",
    "Winnipeg Jets": "Winnipeg",
    "Minnesota Wild": "Minneapolis–Saint Paul",
    "Colorado Avalanche": "Denver",
    "St Louis Blues": "St. Louis",
    "Dallas Stars": "Dallas–Fort Worth",
    "Chicago Blackhawks": "Chicago",
    "Vegas Golden Knights": "Las Vegas",
    "Anaheim Ducks": "Los Angeles",
    "San Jose Sharks": "San Francisco Bay Area",
    "Los Angeles Kings": "Los Angeles",
    "Calgary Flames": "Calgary",
    "Edmonton Oilers": "Edmonton",
    "Vancouver Canucks": "Vancouver",
    "Arizona Coyotes": "Phoenix",
    "Phoenix Coyotes": "Phoenix",
}

In [161]:
nhl_team_to_city_df = pd.DataFrame(list(nhl_team_to_city.items()), columns = ['team','city'])

In [162]:
nhl_df = nhl_df.merge(nhl_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')

In [163]:
nhl_df = nhl_df.groupby(['city','year']).mean()['NHL'].reset_index()

In [164]:
nhl_df

,city,year,NHL
0,Boston,2014,0.739726
1,Boston,2015,0.602941
2,Boston,2017,0.581005
3,Boston,2018,0.714286
4,Buffalo,2014,0.291667
...,...,...,...
104,"Washington, D.C.",2018,0.653333
105,Winnipeg,2014,0.513889
106,Winnipeg,2015,0.623188
107,Winnipeg,2017,0.503153


In [165]:
nba_df['team'] = nba_df['team'].str.split('*').str[0].str.split('(').str[0].str.strip()

In [18]:
#nba_df['team2'] = nba_df['team2'].apply(lambda x: x.split(" (")[0])

In [166]:
nba_df

,team,W,L,W/L%,GB,PS/G,PA/G,SRS,year,League
0,Toronto Raptors,59,23,0.72,—,111.7,103.9,7.29,2018,NBA
1,Boston Celtics,55,27,0.6709999999999999,4.0,104.0,100.4,3.23,2018,NBA
2,Philadelphia 76ers,52,30,0.634,7.0,109.8,105.3,4.3,2018,NBA
3,Cleveland Cavaliers,50,32,0.61,9.0,110.9,109.9,0.59,2018,NBA
4,Indiana Pacers,48,34,0.585,11.0,105.6,104.2,1.18,2018,NBA
...,...,...,...,...,...,...,...,...,...,...
157,San Antonio Spurs,62,20,.756,—,105.4,97.6,8.00,2014,NBA
158,Houston Rockets,54,28,.659,8.0,107.7,103.1,5.06,2014,NBA
159,Memphis Grizzlies,50,32,.610,12.0,96.1,94.6,2.18,2014,NBA
160,Dallas Mavericks,49,33,.598,13.0,104.8,102.4,2.91,2014,NBA


In [168]:
print(nba_df['team'].tolist())

['Toronto Raptors', 'Boston Celtics', 'Philadelphia 76ers', 'Cleveland Cavaliers', 'Indiana Pacers', 'Miami Heat', 'Milwaukee Bucks', 'Washington Wizards', 'Detroit Pistons', 'Charlotte Hornets', 'New York Knicks', 'Brooklyn Nets', 'Chicago Bulls', 'Orlando Magic', 'Atlanta Hawks', 'Houston Rockets', 'Golden State Warriors', 'Portland Trail Blazers', 'Oklahoma City Thunder', 'Utah Jazz', 'New Orleans Pelicans', 'San Antonio Spurs', 'Minnesota Timberwolves', 'Denver Nuggets', 'Los Angeles Clippers', 'Los Angeles Lakers', 'Sacramento Kings', 'Dallas Mavericks', 'Memphis Grizzlies', 'Phoenix Suns', 'Boston Celtics', 'Cleveland Cavaliers', 'Toronto Raptors', 'Washington Wizards', 'Atlanta Hawks', 'Milwaukee Bucks', 'Indiana Pacers', 'Chicago Bulls', 'Miami Heat', 'Detroit Pistons', 'Charlotte Hornets', 'New York Knicks', 'Orlando Magic', 'Philadelphia 76ers', 'Brooklyn Nets', 'Golden State Warriors', 'San Antonio Spurs', 'Houston Rockets', 'Los Angeles Clippers', 'Utah Jazz', 'Oklahoma Cit

In [169]:
nba_team_to_city = {
    "Toronto Raptors": "Toronto",
    "Boston Celtics": "Boston",
    "Philadelphia 76ers": "Philadelphia",
    "Cleveland Cavaliers": "Cleveland",
    "Indiana Pacers": "Indianapolis",
    "Miami Heat": "Miami–Fort Lauderdale",
    "Milwaukee Bucks": "Milwaukee",
    "Washington Wizards": "Washington, D.C.",
    "Detroit Pistons": "Detroit",
    "Charlotte Hornets": "Charlotte",
    "Charlotte Bobcats": "Charlotte",
    "New York Knicks": "New York City",
    "Brooklyn Nets": "New York City",
    "Chicago Bulls": "Chicago",
    "Orlando Magic": "Orlando",
    "Atlanta Hawks": "Atlanta",
    "Houston Rockets": "Houston",
    "Golden State Warriors": "San Francisco Bay Area",
    "Portland Trail Blazers": "Portland",
    "Oklahoma City Thunder": "Oklahoma City",
    "Utah Jazz": "Salt Lake City",
    "New Orleans Pelicans": "New Orleans",
    "San Antonio Spurs": "San Antonio",
    "Minnesota Timberwolves": "Minneapolis–Saint Paul",
    "Denver Nuggets": "Denver",
    "Los Angeles Clippers": "Los Angeles",
    "Los Angeles Lakers": "Los Angeles",
    "Sacramento Kings": "Sacramento",
    "Dallas Mavericks": "Dallas–Fort Worth",
    "Memphis Grizzlies": "Memphis",
    "Phoenix Suns": "Phoenix",
}

In [170]:
nba_team_to_city_df = pd.DataFrame(list(nba_team_to_city.items()), columns = ['team','city'])

In [171]:
nba_df = nba_df.merge(nba_team_to_city_df, how = 'left', left_on = 'team', right_on = 'team')

In [172]:
nba_excl_list = ['Atlantic Division', 'Central Division','Southeast Division','Northwest Division','Pacific Division',
                'Southwest Division']
nba_df = nba_df[~nba_df['team'].isin(nba_excl_list)]

In [173]:
nba_df['W/L%']=nba_df['W/L%'].astype(float)
nba_df['NBA'] = nba_df['W/L%']

In [174]:
nba_df=nba_df.groupby(['city','year']).mean()['NBA'].reset_index()


In [133]:
# merge with cities
base_df = base_df.merge(cities, how = 'outer', on = 'city')

In [175]:
# merge with MLB
base_df = base_df.merge(mlb_df, how = 'outer', on = ['city', 'year'])

In [178]:
# merge with nhl_df
base_df = base_df.merge(nhl_df, how = 'outer', on = ['city', 'year'])

In [180]:
# merge with nba_df
base_df = base_df.merge(nba_df, how = 'outer', on = ['city', 'year'])

In [181]:
base_df

,city,year,NFL,MLB,NHL,NBA
0,Atlanta,2014,0.375,0.488,NaN,0.463
1,Atlanta,2015,0.500,0.414,NaN,0.732
2,Atlanta,2016,0.688,0.422,NaN,0.585
3,Atlanta,2017,0.625,0.444,NaN,0.524
4,Atlanta,2018,0.438,0.556,NaN,0.293
...,...,...,...,...,...,...
143,"Washington, D.C.",2014,0.250,0.593,0.558824,0.537
144,"Washington, D.C.",2015,0.563,0.512,0.633803,0.561
145,"Washington, D.C.",2016,0.531,0.586,NaN,0.500
146,"Washington, D.C.",2017,0.438,0.599,0.750000,0.598


In [187]:
def sports_team_performance():
    # YOUR CODE HERE
    
    # Note: p_values is a full dataframe, so df.loc["NFL","NBA"] should be the same as df.loc["NBA","NFL"] and
    # df.loc["NFL","NFL"] should return np.nan
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    p_values = pd.DataFrame({k:np.nan for k in sports}, index=sports)
    for sport1 in sports:
        for sport2 in sports:
            if sport2!= sport1:
                df = base_df[[sport1,sport2]]
                df = df.dropna()
                p_val = stats.ttest_rel(df[sport1], df[sport2])[1]
                p_values.loc[sport1, sport2] = p_val
        return p_values

In [188]:
sports_team_performance()

,NFL,NBA,NHL,MLB
NFL,NaN,0.746734,0.002089,0.425282
NBA,NaN,NaN,NaN,NaN
NHL,NaN,NaN,NaN,NaN
MLB,NaN,NaN,NaN,NaN
